In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    upper,
    to_date,
    date_format,
    year,
    month,
    dayofmonth,
)


BRONZE_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/bronze/simulated/trades/"
)

SILVER_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/silver/simulated/trades/"
)

CHECKPOINT_PATH = (
    "s3a://kafka-spark-stock-project-kris/"
    "stock-market/checkpoints/silver_simulated_trades/"
)


def main():

    spark = (
        SparkSession.builder
        .appName("StockMarketSilverTrades")
        .config("spark.sql.session.timeZone", "UTC")
        .config(
            "spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider",
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    # -----------------------------------------------------
    # Get schema from existing Bronze Parquet dataset
    # -----------------------------------------------------

    bronze_schema = (
        spark.read
        .parquet(BRONZE_PATH)
        .schema
    )

    # -----------------------------------------------------
    # Stream new Bronze files
    # -----------------------------------------------------

    bronze_stream = (
        spark.readStream
        .schema(bronze_schema)
        .option("maxFilesPerTrigger", 20)
        .parquet(BRONZE_PATH)
    )

    # -----------------------------------------------------
    # Bronze -> Silver transformations
    # -----------------------------------------------------

    silver_df = (
        bronze_stream

        # Standardize symbol
        .withColumn(
            "symbol",
            upper(col("symbol"))
        )

        # Keep deduplication state bounded while allowing
        # reasonably late arriving trade events.
        .withWatermark(
            "event_timestamp",
            "30 minutes"
        )

        .dropDuplicates(["event_id"])

        # Business/analytics fields
        .withColumn(
            "trade_date",
            to_date(col("event_timestamp"))
        )

        .withColumn(
            "trade_time",
            date_format(
                col("event_timestamp"),
                "HH:mm:ss.SSS"
            )
        )

        # Silver is partitioned by EVENT time,
        # not ingestion time like Bronze.
        .withColumn(
            "year",
            year(col("event_timestamp"))
        )

        .withColumn(
            "month",
            month(col("event_timestamp"))
        )

        .withColumn(
            "day",
            dayofmonth(col("event_timestamp"))
        )

        .select(
            "event_id",
            "symbol",
            "price",
            "volume",
            "trade_conditions",

            "event_timestamp",
            "ingestion_timestamp",
            "processing_timestamp",

            "trade_date",
            "trade_time",

            "source",
            "schema_version",

            # Kafka lineage
            "message_key",
            "topic",
            "partition",
            "offset",
            "kafka_timestamp",

            "year",
            "month",
            "day",
        )
    )

    # -----------------------------------------------------
    # Write Silver
    # -----------------------------------------------------

    query = (
        silver_df.writeStream
        .format("parquet")
        .outputMode("append")

        .option(
            "path",
            SILVER_PATH
        )

        .option(
            "checkpointLocation",
            CHECKPOINT_PATH
        )

        .option(
            "compression",
            "snappy"
        )

        .partitionBy(
            "year",
            "month",
            "day"
        )

        # Process everything currently waiting in Bronze
        # and then stop automatically.
        .trigger(availableNow=True)

        .start()
    )

    print("=" * 60)
    print("Silver processing started")
    print(f"Bronze:     {BRONZE_PATH}")
    print(f"Silver:     {SILVER_PATH}")
    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print("=" * 60)

    query.awaitTermination()

    print("=" * 60)
    print("Silver processing completed")
    print("=" * 60)

    spark.stop()


if __name__ == "__main__":
    main()